# This notebook is a continuation of `improving_baseline_model`. In the previous part, a grid search was performed over what is was defined as `set1`. Now, we are going to read what was obtained from that search, look for trends, and see how can we improve the model or which next steps we can take

# Reading `json`output from `improving_baseline_model`


In [1]:
import pandas as pd
from pathlib import Path

json_path = Path('/kaggle/input/notebooks/sebastianvpal/improving-baseline-model/hp_search_results.jsonl')
json_results = pd.read_json(json_path,lines = True)
# Sorting the results: from best to worst
json_results = json_results.sort_values("best_score", ascending = False)

In [2]:
json_results.head(10)

,trial_id,elapsed_s,fold,n_epochs,max_iters,seed,data_parallel,batch_size,method,lr,det_loss_weight,det_neg_weight,det_threshold,best_score,final_edge_loss,final_det_loss,final_test_acc,final_test_recall
13,tier1_013,1112.1,0,4,200,0,False,2,hp_search_tier1,0.00003,1,0.030,0.99,0.928939,0.000620,0.015097,0.998719,0.886219
21,tier1_021,1117.6,0,4,200,0,False,2,hp_search_tier1,0.00003,3,0.010,0.99,0.927859,0.000538,0.006232,0.998931,0.913781
4,tier1_004,1118.9,0,4,200,0,False,2,hp_search_tier1,0.00003,3,0.003,0.40,0.922657,0.000467,0.002466,0.998898,0.923675
14,tier1_014,1111.7,0,4,200,0,False,2,hp_search_tier1,0.00003,10,0.010,0.99,0.916977,0.000656,0.007003,0.998664,0.901060
18,tier1_018,1119.4,0,4,200,0,False,2,hp_search_tier1,0.00003,3,0.003,0.80,0.916358,0.000468,0.002602,0.999213,0.897527
11,tier1_011,1115.5,0,4,200,0,False,2,hp_search_tier1,0.00003,3,0.003,0.99,0.915271,0.000542,0.002548,0.998860,0.913074
19,tier1_019,1111.6,0,4,200,0,False,2,hp_search_tier1,0.00010,1,0.030,0.80,0.914421,0.000663,0.015911,0.998713,0.893993
8,tier1_008,1119.2,0,4,200,0,False,2,hp_search_tier1,0.00010,3,0.003,0.80,0.912062,0.000675,0.002994,0.998892,0.913074
0,tier1_000,1116.9,0,4,200,0,False,2,hp_search_tier1,0.00003,10,0.030,0.40,0.910134,0.000711,0.017160,0.998880,0.901060
12,tier1_012,1108.6,0,4,200,0,False,2,hp_search_tier1,0.00010,3,0.030,0.40,0.906281,0.000692,0.015187,0.998744,0.907420


In [3]:
# sanity-check marginal effect of each param independently
to_chek = ["lr", "det_loss_weight", "det_neg_weight", "det_threshold"]
for param in [*to_chek]:
    print(json_results.groupby(param)["best_score"].agg(["mean", "std", "count"]).sort_values("mean", ascending = False),"\n","--"*20)

             mean       std  count
lr                                
0.00003  0.916309  0.009101      9
0.00010  0.902386  0.009838      8
0.00030  0.894471  0.009578      7 
 ----------------------------------------
                     mean       std  count
det_loss_weight                           
3                0.908980  0.012450     10
1                0.903739  0.016125      7
10               0.901598  0.010540      7 
 ----------------------------------------
                    mean       std  count
det_neg_weight                           
0.003           0.908160  0.011689      8
0.030           0.907502  0.011692      8
0.010           0.900233  0.015340      8 
 ----------------------------------------
                   mean       std  count
det_threshold                           
0.80           0.908605  0.008300      5
0.99           0.906023  0.015809     11
0.40           0.902236  0.011711      8 
 ----------------------------------------


**Observations:** Training for 4 `epochs`and with 200 `max_iters` takes in average *18 minutes* without `data_parallel`

- `best_score` is obtained at **whatever epoch during that trial's training scored highest** and is computed as `test_acc * test_recall`

- Remember that `det_loss_weight` tells me about how strongly detection loss counts relative to edge loss in the combined objective, loss = edge_loss + det_loss_weight * det_loss. It seems that the best value is around 1 to 3

- It looks like the best value was achieved with a `learning rate` of 3e-5.

- For `det_neg_weight` the best value is 3e-3

- The `best_score`among all runs was obtained with a `det_threshold`of 0.80 followed by 0.99

**Comments on the metrics:**
Let's comment briefly on the metrics shown in the previous table, these are: `final_edge_loss`,`final_det_loss`, `final_test_acc`, `final_test_recall`.

- `final_edge_loss` and `final_det_loss` are computed on the `training set` during `train_epoch` and used for **gradient descent**, this two metrics tells me ==what the model is being optimized toward.==
- `final_test_acc` and `final_test_recall` are computed on the `validation set` during `evaluate` and they are used only to **measure and never update weights**. These two metrics tell me ==how well the optimization generalizes==

Let's now remember what is telling me each metric:

- `final_edge_loss`: It tells me how well the model predicts which cell at `t time`links to which cell at `t+1`time (this is the association/tracking task).
- `final_det_loss`: It indetifies whether the detection head correctly indetify `voxels`at true cell centers and stays quiet elsewhere. This metric is calculated in `compute_detection_loss`and called once per frame in the windows (w times). The metric, as for `final_edge_loss`is **cross entropy**. The metric is only computed in voxels with **GT** cell center. Parameters that influence this metric are:
- - `unet_layers`and `unet_out_channels`: The **UNet's** capacity to resolve fine spatial structure --> ability to separate closely-packed cells.
  - `downsample`Coarser spatial downsampling can merge nearby tre cells into the same voxel.
  - `det_neg_weight`This trades false-positive supression against sensitivity. A high value of `det_neg_weight` and the model becomes overly conservative (an error in detection is heavily penalized), and too low and it over detects (a small penalty makes the mold "bold").
  - `learning_rate`
- `final_test_acc`: Once the model generates a **edge-prediction matrix** it computes what did it get right using the validation data (**held-out validation data**) with the model's own detections and not the **GT** nodes as inputs, mimicking real inference conditions.
- `final_test_recall`: Among all the **GT** cells that exist in the validation, what fraction did the detection+matching pipeline actually find. Computed before the **edge predictor** runs. Among the metrics that influence this metric are:
- - `det_threshold` the threshold used to identify a voxel as a node, if you raised lower cells will clear the bar so fewer GT nodes can possible be matched.
  - `pool_kernel_um` too large suppresses nearby true detections as duplicates (one probability peak survived where two cells actually exist, 2 nodes are merged into 1) while too small allow noise to be detected or cells with small probability
  - `max_match_distance`: This value is fixed at `5 um` and it tells me how far a predicted node can be from the **GT** cell and still count as valid prediction.

---
**Note:** If we **decompose** the error into 3 components `detections (nodes,cells)`, `association (edges,links)`, `fragmentation (cell divisions)`, the above metrics are related to 
   - `detection = final_det_loss/final_test_recall`: how well the detection is done in the training and validation datasets.
   - `association = final_edge_loss/final_test_acc`: how well the edges are identify in the training and validation datasets.



# Grid search: Part 2

Now we will perform another grid search, but in this case around the best values ​​obtained in the previous step, so this time the number of parameters to search is smaller, which is particularly important given the computational constraints. 

## Defining input data

In [4]:
from pathlib import Path
import os
DATA_PATH = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
TRAINING_PATH =  DATA_PATH/'train'
TEST_PATH = DATA_PATH/'test'


#------------ Listing folders in train -----------
print("Input contents:", os.listdir(TRAINING_PATH)[:5],"\n")
print(" First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)")

Input contents: ['6bba_2540cd90.geff', '44b6_0b24845f.geff', '44b6_996155de.geff', '44b6_0c582fdc.geff', '6bba_cf35214c.zarr'] 

 First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)


## Modifications to baseline model

In [5]:
import shutil, sys
from pathlib import Path
# --------- Making a copy of the folder to my own working space --------------
# ---------             such that I can edit it     --------------------------

ARTIFACTS_SRC  = Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts")
ARTIFACTS_WORK = Path("/kaggle/working/cellmot-baseline-artifacts")

if not ARTIFACTS_WORK.exists():
    shutil.copytree(ARTIFACTS_SRC, ARTIFACTS_WORK)

# put the writable copy ahead of anything else on the path
sys.path.insert(0, str(ARTIFACTS_WORK))
sys.path.insert(0, str(ARTIFACTS_WORK / "repo/scripts"))   # adjust to wherever train_unet_transformer.py actually sits
sys.path.insert(0, str(ARTIFACTS_WORK / "repo/src"))

In [6]:
needed_dirs = {
    matches[0].parent for name in
    ("train_unet_transformer.py", "tracking_cellmot", "augmentations.py", "dataspec.py")
    if (matches := list(ARTIFACTS_WORK.rglob(name)))
}
needed_dirs

{PosixPath('/kaggle/working/cellmot-baseline-artifacts/repo/scripts'),
 PosixPath('/kaggle/working/cellmot-baseline-artifacts/repo/src')}

### Installing dependencies

In [7]:
import glob
import os
import subprocess
import sys
import shutil

# 1. Find all wheels, but filter OUT numpy wheels to avoid breaking C-extensions
wheels = [
    f for f in glob.glob(f"{ARTIFACTS_SRC}/wheels/*.whl")
    if "numpy" not in os.path.basename(f).lower()
]

# 2. Install only the required non-NumPy wheels without upgrading dependencies
subprocess.run(
    [
        "pip", "install", 
        "--no-index", 
        "--find-links", f"{ARTIFACTS_SRC}/wheels",
        "--no-deps",
        *wheels
    ],
    check=True)

Looking in links: /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/imagecodecs-2026.6.26-cp312-abi3-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/shellingham-1.5.4-py2.py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/bidict-0.23.1-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/requests-2.34.2-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-bas

CompletedProcess(args=['pip', 'install', '--no-index', '--find-links', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels', '--no-deps', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/imagecodecs-2026.6.26-cp312-abi3-manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/shellingham-1.5.4-py2.py3-none-any.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/bidict-0.23.1-py3-none-any.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/requests-2.34.2-py3-none-any.whl', '/kaggle/input/datasets/thibautgolds

### Creating `splits_file`file for the training and test datasets

In [8]:
import json
import random
from pathlib import Path

if not TRAINING_PATH.is_dir():
    raise FileNotFoundError(f"TRAINING_PATH does not exist or is not a directory: {TRAINING_PATH}")

# stems with matching .geff annotation, same convention as the rest of the pipeline
train_pool_stems = sorted(
    p.stem for p in TRAINING_PATH.iterdir()
    if p.is_dir() and p.suffix == ".zarr"
    and (TRAINING_PATH / f"{p.stem}.geff").exists()
)
print(f"{len(train_pool_stems)} videos available in training pool")


MAX_SAMPLES = 10 # define how many datasets will be considered MAX_SAMPLES <= len(train_pool_stems)
VAL_FRACTION = 0.15  # 15% of the total considered data 
n_val = max(1, round(MAX_SAMPLES * VAL_FRACTION))

rng = random.Random(0)          # fixed seed — same split reused across every hp-search trial
shuffled = train_pool_stems.copy()
rng.shuffle(shuffled)
subset = shuffled[:MAX_SAMPLES]

val_stems = sorted(subset[:n_val])
train_stems = sorted(subset[n_val:])

print(f"{len(train_stems)} train / {len(val_stems)} val")
assert set(train_stems).isdisjoint(val_stems)  # sanity check — no leakage between the two

splits = [{"split": 0, "train": train_stems, "test": val_stems}]
# if split = "all" the "train()" function performs 5 folds during training
with open(f"{ARTIFACTS_WORK}/kaggle_train_val_splits.json", "w") as f:
    json.dump(splits, f, indent=2)

199 videos available in training pool
8 train / 2 val


In [9]:
#---------- Reading back the json file just created ------------
with open(f"{ARTIFACTS_WORK}/kaggle_train_val_splits.json", "r") as f:
    json_file = json.load(f)

json_file

[{'split': 0,
  'train': ['44b6_1574802b',
   '44b6_d5e7d891',
   '6bba_2312ac41',
   '6bba_5c824876',
   '6bba_7af54fde',
   '6bba_7b5d3b2c',
   '6bba_afb141ff',
   '6bba_d1acb6ff'],
  'test': ['44b6_d754aa59', '6bba_268e1230']}]

### Applying modifications to the baseline code

Working now on the folder located in my `work` folder

In [10]:
REPO_DIR = ARTIFACTS_WORK / "repo"
target = REPO_DIR / "scripts" / "train_unet_transformer.py"  # path to the training script
src = target.read_text()

# 1. thread det_threshold through train_epoch
src = src.replace(
    "def train_epoch(\n    model: UNetNodeTransformer,\n    loader: DataLoader,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    det_loss_weight: float = 0.1,\n    det_neg_weight: float = 0.1,\n    max_iters: int | None = None,\n    pool_kernel_um: float = 5.0,\n) -> tuple[float, float]:",
    "def train_epoch(\n    model: UNetNodeTransformer,\n    loader: DataLoader,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    det_loss_weight: float = 0.1,\n    det_neg_weight: float = 0.1,\n    max_iters: int | None = None,\n    pool_kernel_um: float = 5.0,\n    det_threshold: float = 0.3,\n) -> tuple[float, float]:"
)
src = src.replace(
    "                voxel_size=voxel_size,\n                pool_kernel_um=pool_kernel_um,\n                frame_index=i, window_size=W,\n            )\n            unet_feat = model._index_features(\n                unet_out[:, i], det_c, det_m,\n            )\n            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n\n        # --- 4. Per-pair edge prediction",
    "                voxel_size=voxel_size,\n                pool_kernel_um=pool_kernel_um,\n                det_threshold=det_threshold,\n                frame_index=i, window_size=W,\n            )\n            unet_feat = model._index_features(\n                unet_out[:, i], det_c, det_m,\n            )\n            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n\n        # --- 4. Per-pair edge prediction"
)

# 2. train() returns metrics instead of just the model
src = src.replace(
    "    print(f\"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}\")\n    if save_path.exists():",
    "    print(f\"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}\")\n    metrics = {\"best_score\": best_score, \"final_edge_loss\": edge_loss,\n               \"final_det_loss\": det_loss, \"final_test_acc\": test_acc,\n               \"final_test_recall\": test_recall}\n    if save_path.exists():"
)
src = src.replace(
    "        model.load_state_dict(state)\n    return model",
    "        model.load_state_dict(state)\n    return model, metrics"
)

target.write_text(src)

49486

In [11]:
REPO_DIR = ARTIFACTS_WORK / "repo"

target = REPO_DIR / "scripts" / "train_unet_transformer.py"  # same path as before
src = target.read_text()

# 1. add full_checkpoint to train()'s signature
src = src.replace(
    "    pool_kernel_um: float = 5.0,\n    data_parallel: bool = True,\n) -> UNetNodeTransformer:",
    "    pool_kernel_um: float = 5.0,\n    data_parallel: bool = True,\n    full_checkpoint: Path | None = None,\n    det_threshold: float = 0.3,\n) -> UNetNodeTransformer:"
)

# 2. load it right after model construction, BEFORE any DataParallel wrapping
#    (checkpoint keys are unwrapped "unet.*", wrapping would change them to "unet.module.*")
src = src.replace(
    "    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=unet_out_channels,\n        pos_feat_dim=pos_feat_dim,\n    ).to(device)\n",
    "    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=unet_out_channels,\n        pos_feat_dim=pos_feat_dim,\n    ).to(device)\n\n"
    "    if full_checkpoint is not None:\n"
    "        ckpt_state = torch.load(full_checkpoint, map_location=device, weights_only=True)\n"
    "        missing, unexpected = model.load_state_dict(ckpt_state, strict=False)\n"
    "        print(f\"  Full checkpoint loaded from {full_checkpoint}: \"\n"
    "              f\"{len(missing)} missing, {len(unexpected)} unexpected\", flush=True)\n"
    "        if missing or unexpected:\n"
    "            print(f\"    missing (sample): {missing[:5]}\", flush=True)\n"
    "            print(f\"    unexpected (sample): {unexpected[:5]}\", flush=True)\n"
)

# 2. pass it into the train_epoch(...) call inside the epoch loop
src = src.replace(
    "        edge_loss, det_loss = train_epoch(\n"
    "            model, train_loader, optimizer, device, det_loss_weight, det_neg_weight,\n"
    "            max_iters=max_iters, pool_kernel_um=pool_kernel_um,\n"
    "        )",
    "        edge_loss, det_loss = train_epoch(\n"
    "            model, train_loader, optimizer, device, det_loss_weight, det_neg_weight,\n"
    "            max_iters=max_iters, pool_kernel_um=pool_kernel_um, det_threshold=det_threshold,\n"
    "        )"
)

# 3. pass it into the evaluate(...) call inside the epoch loop
src = src.replace(
    "        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um)",
    "        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um, det_threshold=det_threshold)"
)


target.write_text(src)

# sanity checks — confirm both edits actually matched before trusting them
assert "full_checkpoint: Path | None = None," in src
assert "Full checkpoint loaded from" in src
assert "det_threshold: float = 0.3,\n) -> UNetNodeTransformer:" in src
assert "pool_kernel_um=pool_kernel_um, det_threshold=det_threshold,\n        )" in src
assert "evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um, det_threshold=det_threshold)" in src

In [12]:
# 1. add det_threshold to evaluate()'s signature
src = src.replace(
    "def evaluate(\n"
    "    model: UNetNodeTransformer,\n"
    "    loader: DataLoader,\n"
    "    device: torch.device,\n"
    "    pool_kernel_um: float = 5.0,\n"
    ") -> tuple[float, float, float]:",
    "def evaluate(\n"
    "    model: UNetNodeTransformer,\n"
    "    loader: DataLoader,\n"
    "    device: torch.device,\n"
    "    pool_kernel_um: float = 5.0,\n"
    "    det_threshold: float = 0.3,\n"
    ") -> tuple[float, float, float]:"
)

# 2. pass it into detect_and_match(...) inside evaluate()'s loop
src = src.replace(
    "            det_c, det_p, det_m, matches = detect_and_match(\n"
    "                det_logits[i], coords[:, i], masks[:, i],\n"
    "                image_shape,\n"
    "                voxel_size=voxel_size,\n"
    "                pool_kernel_um=pool_kernel_um,\n"
    "                frame_index=i, window_size=W,\n"
    "            )\n"
    "            unet_feat = model._index_features(\n"
    "                unet_out[:, i], det_c, det_m,\n"
    "            )\n"
    "            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n"
    "\n"
    "            # Node recall:",
    "            det_c, det_p, det_m, matches = detect_and_match(\n"
    "                det_logits[i], coords[:, i], masks[:, i],\n"
    "                image_shape,\n"
    "                voxel_size=voxel_size,\n"
    "                pool_kernel_um=pool_kernel_um,\n"
    "                det_threshold=det_threshold,\n"
    "                frame_index=i, window_size=W,\n"
    "            )\n"
    "            unet_feat = model._index_features(\n"
    "                unet_out[:, i], det_c, det_m,\n"
    "            )\n"
    "            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n"
    "\n"
    "            # Node recall:"
)

target.write_text(src)
assert "det_threshold: float = 0.3,\n) -> tuple[float, float, float]:" in src

In [13]:
lines = target.read_text().splitlines(keepends=True)

# line 1201 in your grep output is 1-indexed; list index is 1200
assert "Best score (acc" in lines[1200], f"unexpected content at index 1200: {lines[1200]!r}"

# metrics_block = (
#     '    metrics = {"best_score": best_score, "final_edge_loss": edge_loss,\n'
#     '               "final_det_loss": det_loss, "final_test_acc": test_acc,\n'
#     '               "final_test_recall": test_recall}\n'
# )

metrics_block = (
    '    metrics = {"best_score": best_score, "final_edge_loss": edge_loss,\n'
    '               "final_det_loss": det_loss, "final_test_acc": test_acc,\n'
    '               "final_test_recall": test_recall,\n'
    '               "best_test_acc": best_test_acc, "best_test_recall": best_test_recall}\n'
)


lines.insert(1201, metrics_block)  # insert right after the print line
target.write_text("".join(lines))
print("inserted")


inserted


In [14]:
REPO_DIR = ARTIFACTS_WORK / "repo"

target = REPO_DIR / "scripts" / "train_unet_transformer.py"  # same path as before
src = target.read_text()

old = (
    "        if is_best:\n"
    "            best_score = score\n"
)
new = (
    "        if is_best:\n"
    "            best_score = score\n"
    "            best_test_acc, best_test_recall = test_acc, test_recall\n"
)

#to check my text is well constructed with respect to the .py file
assert src.count(old) == 1, f"expected exactly 1 match, found {src.count(old)}"

src = src.replace(old, new)
target.write_text(src)

# verify against the file, not just the in-memory string
current = target.read_text()
assert "best_test_acc, best_test_recall = test_acc, test_recall" in current
print("confirmed on disk")

confirmed on disk


In [15]:
import inspect
from train_unet_transformer import train
print(inspect.getsource(train).count("best_test_acc"))  # should be >= 2 (assignment + dict entry)

3


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


### Run configuration

In [16]:
METHOD = "unet_transformer"

# --------------- Reading weights from the baseline model ---------------------------------------
# in the test set, the baseline model scored 0.8, let's see if we can improve this
# Model checkpoint (relative to the repo, or an absolute path to your own).
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"

### Refined sweep of set 1

In [17]:
import itertools
import json
import time
from pathlib import Path
import gc
import torch
import random

from train_unet_transformer import train

RESULTS_PATH = Path("hp_search_results_part2.jsonl")
random.seed(0)
n_random_trials = 12  #approx 4 hours with 12 trials



def log_result(record: dict) -> None:
    with open(RESULTS_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

def run_trial(trial_id: str, **kwargs) -> dict:
    t0 = time.monotonic()
    try:
        model, metrics = train(**kwargs)
    finally:
        # ensure cleanup happens even if this trial itself OOMs
        if "model" in dir():
            del model
        gc.collect()
        torch.cuda.empty_cache()
    elapsed = time.monotonic() - t0
    record = {
        "trial_id": trial_id,
        "elapsed_s": round(elapsed, 1),
        **{k: (str(v) if isinstance(v, Path) else v) for k, v in kwargs.items()
           if k not in ("data_dir", "splits_file", "unet_layers", "full_checkpoint")},
        **metrics,
    }
    log_result(record)
    print(f"[{trial_id}] score={metrics['best_score']:.4f}  ({elapsed:.0f}s)")
    return record


FULL_CHECKPOINT = ARTIFACTS_SRC / "weights" / METHOD / "split_0" / "edge_predictor_best.pth"
assert FULL_CHECKPOINT.exists(), f"not found: {FULL_CHECKPOINT}"

FIXED = dict(
    data_dir=Path(DATA_PATH)/"train",
    splits_file=ARTIFACTS_WORK/"kaggle_train_val_splits.json",
    fold=0,
    n_epochs=4,
    max_iters=200,
    seed=0,
    data_parallel=False, #set it to False given the small batch_size
    batch_size=2,
    method="hp_search_tier1",
    full_checkpoint=FULL_CHECKPOINT,   # <-- warm-start every trial from the 0.80 baseline
)


tier1b_space = {
    "lr": [1e-5, 3e-5, 5e-5],              # bracket below and around the current winner
    "det_loss_weight": [2, 3, 4],          # move araound the optimal = 3
    "det_neg_weight": [0.003, 0.01, 0.03], #similar best score: 0.003 -> best score 0.908 and 0.03 -> best score 0.907
    "det_threshold": [0.8, 0.92, 0.99],      # move from 0.80 to 0.99
}


keys = list(tier1b_space.keys())
combos = list(itertools.product(*tier1b_space.values()))
random.shuffle(combos)


#for i, combo in enumerate(itertools.product(*tier1_space.values())):
# for i, combo in enumerate(combos[:1]): # to test one sample
for i, combo in enumerate(combos[:n_random_trials]):
    kwargs = dict(zip(keys, combo))
    run_trial(f"tier1_{i:03d}", **FIXED, **kwargs)
    print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
    print(torch.cuda.memory_reserved() / 1e9, "GB reserved")
    #break

Fold 0: 8 train, 2 test
Loading train (8 datasets)...


train: 100%|██████████| 8/8 [00:05<00:00,  1.44it/s]

  train done: 787 windows total
Loading test (2 datasets)...



test: 100%|██████████| 2/2 [00:01<00:00,  1.77it/s]

  test done: 164 windows total
max_nodes=17


...

Training:  75%|███████▌  | 3/4 [16:55<04:14, 254.21s/it, acc=0.9992, det=0.0027, edge=0.0005]

  Epoch   3/4 | edge=0.0005 | det=0.0027 | test_loss=0.0014 | acc=0.9992 | recall=0.9011 | best=0.9004 * | train=231.5s test=20.4s


Training: 100%|██████████| 4/4 [16:55<00:00, 253.79s/it, acc=0.9992, det=0.0027, edge=0.0005]


Best score (acc*recall): 0.9004, saved to /kaggle/working/cellmot-baseline-artifacts/repo/weights/hp_search_tier1/split_0/edge_predictor_best.pth


[tier1_011] score=0.9004  (1022s)
0.019136512 GB allocated
0.148897792 GB reserved


In [18]:
import pandas as pd

df = pd.read_json("hp_search_results_part2.jsonl", lines=True)
df = df.sort_values("best_score", ascending=False)
print(df.head(10))



     trial_id  elapsed_s  fold  n_epochs  max_iters  seed  data_parallel  \
3   tier1_003     1006.2     0         4        200     0          False   
0   tier1_000     1022.8     0         4        200     0          False   
1   tier1_001     1010.8     0         4        200     0          False   
7   tier1_007     1006.8     0         4        200     0          False   
2   tier1_002     1007.4     0         4        200     0          False   
5   tier1_005     1009.2     0         4        200     0          False   
6   tier1_006     1014.6     0         4        200     0          False   
8   tier1_008     1021.0     0         4        200     0          False   
11  tier1_011     1021.9     0         4        200     0          False   
9   tier1_009     1013.6     0         4        200     0          False   

    batch_size           method       lr  det_loss_weight  det_neg_weight  \
3            2  hp_search_tier1  0.00005                4           0.010   
0        

In [19]:
# sanity-check marginal effect of each param independently
for param in ["lr", "det_loss_weight", "det_neg_weight", "det_threshold"]:
    print(df.groupby(param)["best_score"].agg(["mean", "std", "count"]))

             mean       std  count
lr                                
0.00001  0.901675  0.012126      5
0.00003  0.908631  0.006811      4
0.00005  0.910696  0.013560      3
                     mean       std  count
det_loss_weight                           
2                0.897297  0.009706      3
3                0.901955  0.007088      4
4                0.915054  0.007758      5
                    mean       std  count
det_neg_weight                           
0.003           0.897603  0.007253      5
0.010           0.916179  0.009892      3
0.030           0.909608  0.007322      4
                   mean       std  count
det_threshold                           
0.80           0.910936  0.013056      5
0.92           0.897898  0.010177      3
0.99           0.906652  0.004932      4
